# 模块与模块解析

学习目标：能区分类型导入和运行时依赖，为 Node.js 选择匹配的模块配置，并核对 ESM 与 CommonJS 的实际导入。

前置知识：JavaScript 模块、动态导入、Promise、类型和值、文件相对路径和 package.json。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ESM 与 CommonJS，开启 strict；另启用 verbatimModuleSyntax。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/17-modules-resolution/。

1. [model.ts](scripts/17-modules-resolution/model.ts)：配套实现与示例。
2. [barrel.ts](scripts/17-modules-resolution/barrel.ts)：配套实现与示例。
3. [main.ts](scripts/17-modules-resolution/main.ts)：配套实现与示例。
4. [esm-value.mts](scripts/17-modules-resolution/esm-value.mts)：配套实现与示例。
5. [legacy.cts](scripts/17-modules-resolution/legacy.cts)：配套实现与示例。
6. [cjs-consumer.cts](scripts/17-modules-resolution/cjs-consumer.cts)：配套实现与示例。
7. [scope-module-a.ts](scripts/17-modules-resolution/scope-module-a.ts)：配套实现与示例。
8. [scope-module-b.ts](scripts/17-modules-resolution/scope-module-b.ts)：配套实现与示例。
9. [tsconfig.json](scripts/17-modules-resolution/tsconfig.json)：本章独立项目配置。
10. [type-errors.ts](scripts/17-modules-resolution/type-errors.ts)、[tsconfig.errors.json](scripts/17-modules-resolution/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:17
```

Step 2：生成本章 JavaScript。

```bash
npm run build:17
```

Step 3：运行本章正常示例。

```bash
npm run run:17
# 正常退出；各段预期输出见代码注释。
```

同一文件的片段按正文顺序衔接，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 模块的类型导出与值导出

模块能同时导出类型与值。interface 和 type 只参与检查，函数和对象则可以成为运行时导出。不要因为名称能在编辑器里导入，就假设生成的 JavaScript 中一定有这个名称。

本章使用最近的课程 package.json 中的 type: module；model.ts 因此按 ESM 处理。

对应 [model.ts](scripts/17-modules-resolution/model.ts)。

```typescript
export interface Entry { title: string; hours: number }
export function total(entry: Entry): number { return entry.hours * 2; }
export const unit = "小时";
```

## 2 import type 与 export type

沿两条通道阅读再导出：Entry 提供检查契约，total 与 unit 提供可运行的值。

import type 和 export type 明确擦除类型依赖。下面 barrel.ts 转出类型 Entry，同时转出实际值 total；main.ts 分别导入它们。若某模块需要执行初始化副作用，必须使用真正的值导入或单独的副作用导入，不能依赖 import type。

本章显式启用 verbatimModuleSyntax：带 type 的导入导出按规则擦除，其余语法保留；它也帮助发现把只存在于类型层的名称错误写成普通导入的情况。

![同一模块的类型依赖与值依赖。本章 verbatimModuleSyntax 下，显式 type 导入导出在生成时擦除。](image/illustration/17-01-type-value-module.svg)

图示说明：依据类型导入擦除与值导入规则自绘，虚线表示检查期依赖，实线表示运行值依赖；图省略加载顺序，不把类型检查等同于宿主执行。

下面先找 export type，再找普通 export；继续阅读 main.ts 时，检查调用 total 的导入是否保留到 JavaScript。

对应 [barrel.ts](scripts/17-modules-resolution/barrel.ts)。

```typescript
export type { Entry } from "./model.js";
export { total, unit } from "./model.js";
```

## 3 类型查询 import() 与动态 import()

类型位置的 import() 可以访问另一模块的类型，不产生运行时加载。typeof import() 则取得模块值导出的静态形状。表达式位置的动态 import() 真正加载模块并返回 Promise，两者不能互换。

下面类型声明没有发起动态导入；显式 await import() 才执行加载。导入字符串写输出文件的 .js 扩展名，TypeScript 在检查时对应找到 .ts 源文件。

对应 [main.ts](scripts/17-modules-resolution/main.ts)。

```typescript
import type { Entry } from "./barrel.js";
import { total, unit } from "./barrel.js";
import legacy from "./legacy.cjs";
type ImportedEntry = import("./model.js").Entry;
type ModelModule = typeof import("./model.js");
const entry: Entry = { title: "模块", hours: 2 };
const copy: ImportedEntry = entry;
const loaded: ModelModule = await import("./model.js");
console.log(total(copy), unit, loaded.unit, legacy.double(3));
// 预期输出：4 小时 小时 6
```

## 4 扩展名和包的 type

NodeNext 按文件形式及包边界判断输出格式，并不表示所有输出都是 ESM。声明文件也具有模块格式，需要与相应实现一致。

| 源文件扩展名 | 中文名称／含义 | 输出扩展名 | 格式依据 |
| --- | --- | --- | --- |
| .ts | 普通 TypeScript 源文件 | .js | 最近 package.json 的 type |
| .mts | 显式 ESM 源文件 | .mjs | 固定 ESM |
| .cts | 显式 CommonJS 源文件 | .cjs | 固定 CommonJS |

Node 的 ESM 相对导入需要完整扩展名。写 .js、.mjs、.cjs 是针对编译后的布局；更换输出方式时要重新核对。下面 .mts 无论所在包的 type 如何，都输出 ESM。

对应 [esm-value.mts](scripts/17-modules-resolution/esm-value.mts)。

```typescript
export const esmValue = 17;
```

## 5 从 ESM 使用 CommonJS

Node.js 24.11.0 将 CommonJS 的 module.exports 作为 ESM 默认导入的值。具名导出依赖宿主的静态分析，类型声明认可某名称并不能保证 Node 能提取它；这里使用默认导入读取对象成员。

在 verbatimModuleSyntax 下，.cts 不能随意使用会要求转换为 CommonJS 的 ESM 导出语法。本例用 TypeScript 的 export = 描述 module.exports，生成代码后由 main.ts 的默认导入消费。

对应 [legacy.cts](scripts/17-modules-resolution/legacy.cts)。

```typescript
const legacy = { double(value: number) { return value * 2; } };
export = legacy;
```

## 6 从 CommonJS 加载 ESM

动态 import() 可以从 CommonJS 加载 ESM，并等待其完成。Node.js 24.11.0 的 require(ESM) 则要求整个模块依赖图没有顶层 await；TypeScript 并不完整检查这个运行条件。

本例动态导入自己的 .mts 输出，没有网络与延时。成功加载后在 then 中读取值；这里不添加仅用于打印并改写退出状态的失败包装。

对应 [cjs-consumer.cts](scripts/17-modules-resolution/cjs-consumer.cts)。

```typescript
import("./esm-value.mjs").then(module => console.log("CJS->ESM", module.esmValue));
// 预期输出：CJS->ESM 17
```

## 7 模块作用域与全局脚本

模块的顶层名称属于该模块。显式 export {} 可将原本没有导入导出的文件表明为模块，两个文件里同名的局部变量不会合并。

传统脚本的顶层声明可能进入共享全局作用域。不能仅按“有没有 import”判断所有配置：NodeNext 与 moduleDetection: auto 还会考虑包的 type。本例显式加 export {}，其局部类型不污染其他章节；环境声明与全局扩充另有规则。

对应 [scope-module-a.ts](scripts/17-modules-resolution/scope-module-a.ts)。

```typescript
const localName = "甲";
export {};
```

## 8 另一个独立模块

第二个模块使用同一个局部变量名。正常项目同时检查两个文件，仍不会把这两个声明当作同一个变量。

对应 [scope-module-b.ts](scripts/17-modules-resolution/scope-module-b.ts)。

```typescript
const localName = "乙";
export {};
```

## 9 module 与 moduleResolution 配套选择

module 告诉编译器输出格式及宿主的模块语义；moduleResolution 告诉它如何按导入说明符查找实现与类型。NodeNext 配套用于本章直接由 Node 执行的产物，还需固定宿主版本，因为 NodeNext 会随编译器版本演进。

bundler 解析用于由兼容打包器处理模块的项目，可接受一些 Node ESM 直接运行不接受的无扩展名路径。它通常配合 module: preserve 或 esnext，不能仅为消除 Node 项目的扩展名错误而切换。本章没有打包器依赖，因此不声称做了打包运行。

完整 tsconfig.json 使用显式 NodeNext、strict、ES2025 与 types: node，并通过 files 隔离章节。不要在 TypeScript 7 新项目中使用已移除的 moduleResolution: node10 或 classic。

## 10 检查类型边界

下面的 [type-errors.ts](scripts/17-modules-resolution/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
import type { total } from "./model.js";
import { Entry } from "./model.js"; // verbatimModuleSyntax 下 Entry 必须写成类型导入。
import { unit } from "./model"; // Node ESM 相对导入缺少输出扩展名。
total({ title: "错误", hours: 1 }); // import type 不提供运行时 total。
// 预期诊断包含：TS1484, TS2835, TS1361。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:17
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

类型依赖、输出语法、模块查找与宿主加载是相关但不同的步骤。用实际输出扩展名和包的 type 对齐 NodeNext，并对 ESM/CommonJS 互操作执行验证。

## 练习

1. 给 Entry 新增可选 note，用 import type 消费；检查生成的 main.js 中没有 Entry 导入，再实际运行。

2. 在 .mts 中新增一个值导出，从 cjs-consumer.cts 动态导入后打印；核对 Promise 完成且进程正常退出。

3. 把错误导入 ./model 修正为 ./model.js，确认相应诊断消失；解释为何源码对应的文件仍为 model.ts。

### 提示

1. 在 model.ts 修改接口，barrel.ts 已经转出该类型，不需增加运行时导出。
2. 修改 .mts 的值导出和 then 回调中的读取，重新 build 后再运行。
3. 源码导入说明符描述运行输出路径；类型解析可以对应源文件扩展名。


### 参考解析

1. note?: string 不生成新导出；main.js 中保留 total、unit 等值导入，Entry 被擦除。原正常输出保持不变。
2. 例如新增 extra = 18，在回调打印 module.extra 时应观察到 18；不能只修改源码后运行旧 .cjs。
3. Node 执行的是生成后的 model.js；TypeScript 检查时依据该说明符找到 model.ts 的类型与实现。修正一个导入不会自动修复其他独立的 type 导入误用诊断。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [Modules Reference：类型导入、import()、NodeNext、bundler](https://www.typescriptlang.org/docs/handbook/modules/reference.html)；[Modules Theory：脚本与模块、宿主](https://www.typescriptlang.org/docs/handbook/modules/theory.html)；[verbatimModuleSyntax](https://www.typescriptlang.org/tsconfig/verbatimModuleSyntax.html)。 |
| Node.js 24.11.0 | [Packages：Determining module system](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html#determining-module-system)；[ESM：Mandatory file extensions 与 Interoperability with CommonJS](https://nodejs.org/download/release/v24.11.0/docs/api/esm.html)。 |
| Microsoft Developer Blogs | [TypeScript 7.0：Breaking Changes](https://devblogs.microsoft.com/typescript/announcing-typescript-7-0/)。 |
